# Setup


In [3]:
# Data Preparation
import numpy as np
import pandas as pd
from pathlib import Path

# Data Visualisation
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [4]:
# load files

folder_path = Path('..')
file_path = folder_path / 'Datasets' / 'origin_destination_train_202605.csv'
final_file_path = folder_path / 'Cleaned Datasets' / "Train_Stations_Districts_Cleaned.csv"

dataset = pd.read_csv(file_path)

dataset.head(5)

,YEAR_MONTH,DAY_TYPE,TIME_PER_HOUR,PT_TYPE,ORIGIN_PT_CODE,DESTINATION_PT_CODE,TOTAL_TRIPS
0,2026-05,WEEKENDS/HOLIDAY,19,TRAIN,EW28,CC17/TE9,16
1,2026-05,WEEKDAY,9,TRAIN,PW3,EW1,1
2,2026-05,WEEKDAY,20,TRAIN,PW4,CC27,4
3,2026-05,WEEKENDS/HOLIDAY,11,TRAIN,TE24,CC10/DT26,1
4,2026-05,WEEKENDS/HOLIDAY,16,TRAIN,TE4,DT22,1


In [5]:
# search for missing data

missing = dataset.isnull().sum()

missing

YEAR_MONTH             0
DAY_TYPE               0
TIME_PER_HOUR          0
PT_TYPE                0
ORIGIN_PT_CODE         0
DESTINATION_PT_CODE    0
TOTAL_TRIPS            0
dtype: int64

In [6]:
# no missing so check for duplicates

duplicates = dataset.duplicated().sum()

duplicates

np.int64(0)

# Data Preparation

format would be

    station_code | time | leave | enter | traffic

In [7]:
dataset.head(3)

,YEAR_MONTH,DAY_TYPE,TIME_PER_HOUR,PT_TYPE,ORIGIN_PT_CODE,DESTINATION_PT_CODE,TOTAL_TRIPS
0,2026-05,WEEKENDS/HOLIDAY,19,TRAIN,EW28,CC17/TE9,16
1,2026-05,WEEKDAY,9,TRAIN,PW3,EW1,1
2,2026-05,WEEKDAY,20,TRAIN,PW4,CC27,4


In [8]:
# remove unnecessary columns

dataset = dataset.drop(['YEAR_MONTH', 'DAY_TYPE', 'PT_TYPE'], axis = 1)

dataset.head(3)

,TIME_PER_HOUR,ORIGIN_PT_CODE,DESTINATION_PT_CODE,TOTAL_TRIPS
0,19,EW28,CC17/TE9,16
1,9,PW3,EW1,1
2,20,PW4,CC27,4


In [9]:
regions = {
    'NS10': 'D25',              # Admiralty
    'NS16': 'D20',              # Ang Mo Kio
    'NS17/CC15': 'D20',         # Bishan
    'NS18': 'D13',              # Braddell
    'NS2': 'D23',               # Bukit Batok
    'NS3': 'D23',               # Bukit Gombak
    'DT1/BP6': 'D23',           # Bukit Panjang
    'NS4/BP1': 'D23',           # Choa Chu Kang
    'NS12': 'D27',              # Canberra
    'NS25/EW13': 'D06',         # City Hall
    'NS24/NE6/CC1': 'D09',      # Dhoby Ghaut
    'NS14': 'D27',              # Khatib
    'NS7': 'D25',               # Kranji
    'NS1/EW24': 'D22',          # Jurong East
    'NS27/CE2/TE20': 'D01',     # Marina Bay
    'NS28': 'D01',              # Marina South Pier
    'NS8': 'D25',               # Marsiling
    'NS21/DT11': 'D11',         # Newton
    'NS20': 'D11',              # Novena
    'NS22/TE14': 'D09',         # Orchard
    'NS26/EW14': 'D01',         # Raffles Place
    'NS11': 'D27',              # Sembawang
    'NS23': 'D09',              # Somerset
    'NS19': 'D12',              # Toa Payoh
    'NS5': 'D23',               # Yew Tee
    'NS15': 'D20',              # Yio Chu Kang
    'NS13': 'D27',              # Yishun
    'EW9': 'D14',               # Aljunied
    'EW5': 'D16',               # Bedok
    'EW27': 'D22',              # Boon Lay
    'EW21/CC22': 'D05',         # Buona Vista
    'EW12/DT14': 'D07',         # Bugis
    'EW25': 'D22',              # Chinese Garden
    'CG2': 'D17',               # Changi Airport
    'EW23': 'D05',              # Clementi
    'EW20': 'D03',              # Commonwealth
    'EW22': 'D05',              # Dover
    'EW7': 'D14',               # Eunos
    'CG1/DT35': 'D17',          # Expo
    'EW30': 'D22',              # Gul Circle
    'EW10': 'D14',              # Kallang
    'EW6': 'D14',               # Kembangan
    'EW26': 'D22',              # Lakeside
    'EW11': 'D07',              # Lavender
    'EW16/NE3/TE17': 'D03',     # Outram Park
    'EW1': 'D18',               # Pasir Ris
    'EW8/CC9': 'D14',           # Paya Lebar
    'EW28': 'D22',              # Pioneer
    'EW19': 'D03',              # Queenstown
    'EW18': 'D03',              # Redhill
    'EW3': 'D18',               # Simei
    'EW2/DT32': 'D18',          # Tampines
    'EW4/CG': 'D16',            # Tanah Merah
    'EW15': 'D02',              # Tanjong Pagar
    'EW17': 'D03',              # Tiong Bahru
    'EW31': 'D22',              # Tuas Crescent
    'EW33': 'D22',              # Tuas Link
    'EW32': 'D22',              # Tuas West Road
    'NE9': 'D12',               # Boon Keng
    'NE15': 'D19',              # Buangkok
    'NE4/DT19': 'D01',          # Chinatown
    'NE5': 'D06',               # Clarke Quay
    'NE8': 'D08',               # Farrer Park
    'NE1/CC29': 'D04',          # HarbourFront
    'NE14': 'D19',              # Hougang
    'NE13': 'D19',              # Kovan
    'NE7/DT12': 'D08',          # Little India
    'NE10': 'D13',              # Potong Pasir
    'NE17/PTC': 'D19',          # Punggol
    'NE18': 'D19',              # Punggol Coast
    'NE16/STC': 'D19',          # Sengkang
    'NE12/CC13': 'D12',         # Serangoon
    'NE11': 'D13',              # Woodleigh
    'CC12': 'D13',              # Bartley
    'CE1/DT16': 'D01',          # Bayfront
    'CC19/DT9': 'D10',          # Botanic Gardens
    'CC2': 'D06',               # Bras Basah
    'CC17/TE9': 'D11',          # Caldecott
    'CC8': 'D14',               # Dakota
    'CC3': 'D06',               # Esplanade
    'CC20': 'D10',              # Farrer Road
    'CC25': 'D05',              # Haw Par Villa
    'CC21': 'D10',              # Holland Village
    'CC24': 'D05',              # Kent Ridge
    'CC27': 'D04',              # Labrador Park
    'CC14': 'D12',              # Lorong Chuan
    'CC10/DT26': 'D13',         # MacPherson
    'CC16': 'D20',              # Marymount
    'CC7': 'D14',               # Mountbatten
    'CC5': 'D07',               # Nicoll Highway
    'CC23': 'D05',              # one-north
    'CC26': 'D05',              # Pasir Panjang
    'CC4/DT15': 'D01',          # Promenade
    'CC6': 'D14',               # Stadium
    'DT10/TE11': 'D11',         # Stevens
    'CC11': 'D13',              # Tai Seng
    'CC28': 'D04',              # Telok Blangah
    'DT5': 'D21',               # Beauty World
    'DT29': 'D16',              # Bedok North
    'DT30': 'D16',              # Bedok Reservoir
    'DT21': 'D06',              # Bencoolen
    'DT23': 'D12',              # Bendemeer
    'DT2': 'D23',               # Cashew
    'DT17': 'D01',              # Downtown
    'DT20': 'D06',              # Fort Canning
    'DT24': 'D12',              # Geylang Bahru
    'DT3': 'D23',               # Hillview
    'DT4': 'D23',               # Hume
    'DT22': 'D07',              # Jalan Besar
    'DT28': 'D14',              # Kaki Bukit
    'DT6': 'D21',               # King Albert Park
    'DT25': 'D14',              # Mattar
    'DT13': 'D08',              # Rochor
    'DT7': 'D21',               # Sixth Avenue
    'DT33': 'D18',              # Tampines East
    'DT31': 'D18',              # Tampines West
    'DT8': 'D21',               # Tan Kah Kee
    'DT18': 'D01',              # Telok Ayer
    'DT27': 'D14',              # Ubi
    'DT34': 'D17',              # Upper Changi
    'TE29': 'D16',              # Bayshore
    'TE7': 'D26',               # Bright Hill
    'TE24': 'D15',              # Katong Park
    'TE28': 'D16',              # Siglap
    'TE22': 'D01',              # Gardens by the Bay
    'TE16': 'D03',              # Havelock
    'TE15': 'D09',              # Great World
    'TE18': 'D02',              # Maxwell
    'TE12': 'D10',              # Napier
    'TE13': 'D10',              # Orchard Boulevard
    'TE26': 'D15',              # Marine Parade
    'TE27': 'D15',              # Marine Terrace
    'TE6': 'D26',               # Mayflower
    'TE19': 'D02',              # Shenton Way
    'TE25': 'D15',              # Tanjong Katong
    'TE23': 'D15',              # Tanjong Rhu
    'TE5': 'D26',               # Lentor
    'TE4': 'D26',               # Springleaf
    'TE8': 'D26',               # Upper Thomson
    'NS9/TE2': 'D25',           # Woodlands
    'TE1': 'D25',               # Woodlands North
    'TE3': 'D25',               # Woodlands South
    'BP2': 'D23',               # South View
    'BP3': 'D23',               # Keat Hong
    'BP4': 'D23',               # Teck Whye
    'BP5': 'D23',               # Phoenix
    'BP7': 'D23',               # Petir 
    'BP8': 'D23',               # Pending
    'BP9': 'D23',               # Bangkit  
    'BP10': 'D23',              # Fajar 
    'BP11': 'D23',              # Segar  
    'BP12': 'D23',              # Jelapang  
    'BP13': 'D23',              # Senja
    'SE1': 'D19',               # Compassvale 
    'SE2': 'D19',               # Rumbia
    'SE3': 'D19',               # Bakau  
    'SE4': 'D19',               # Kangkar  
    'SE5': 'D19',               # Ranggung
    'SW1': 'D19',               # Cheng Lim
    'SW2': 'D19',               # Farmway
    'SW3': 'D19',               # Kupang 
    'SW4': 'D19',               # Thanggam 
    'SW5': 'D19',               # Fernvale 
    'SW6': 'D19',               # Layar 
    'SW7': 'D19',               # Tongkang 
    'SW8': 'D19',               # Renjong
    'PE1': 'D19',               # Cove  
    'PE2': 'D19',               # Meridian  
    'PE3': 'D19',               # Coral Edge 
    'PE4': 'D19',               # Riviera  
    'PE5': 'D19',               # Kadaloor  
    'PE6': 'D19',               # Oasis
    'PE7': 'D19',               # Damai 
    'PW1': 'D19',               # Sam Kee  
    'PW2': 'D19',               # Teck Lee 
    'PW3': 'D19',               # Punggol Point 
    'PW4': 'D19',               # Samudera 
    'PW5': 'D19',               # Nibong 
    'PW6': 'D19',               # Sumang 
    'PW7': 'D19',               # Soo Teck  
}

In [10]:
# map the original to the districts

dataset['district_in'] = dataset['DESTINATION_PT_CODE'].map(regions)
dataset['district_out'] = dataset['ORIGIN_PT_CODE'].map(regions)

dataset.head(3)

,TIME_PER_HOUR,ORIGIN_PT_CODE,DESTINATION_PT_CODE,TOTAL_TRIPS,district_in,district_out
0,19,EW28,CC17/TE9,16,D11,D22
1,9,PW3,EW1,1,D18,D19
2,20,PW4,CC27,4,D04,D19


In [11]:
# make a new dataset with origin codes

final_dataset = pd.DataFrame(columns = ['district', 'time', 'leave', 'enter', 'traffic'])

In [12]:
# find types of districts in dataset
grouped_in = dataset.groupby(['district_in', 'TIME_PER_HOUR'])['TOTAL_TRIPS'].sum().reset_index()

grouped_in.head(5)

,district_in,TIME_PER_HOUR,TOTAL_TRIPS
0,D01,0,484
1,D01,5,39868
2,D01,6,125972
3,D01,7,317133
4,D01,8,563664


In [13]:
final_dataset['district'] = grouped_in['district_in'].copy()
final_dataset['time'] = grouped_in['TIME_PER_HOUR'].copy()
final_dataset['enter'] = grouped_in['TOTAL_TRIPS'].copy()

final_dataset.head(5)

,district,time,leave,enter,traffic
0,D01,0,NaN,484,NaN
1,D01,5,NaN,39868,NaN
2,D01,6,NaN,125972,NaN
3,D01,7,NaN,317133,NaN
4,D01,8,NaN,563664,NaN


In [14]:
# check for number of leave

grouped_out = dataset.groupby(['district_out', 'TIME_PER_HOUR'])['TOTAL_TRIPS'].sum().reset_index()

final_dataset['leave'] = grouped_out['TOTAL_TRIPS'].copy()

final_dataset

,district,time,leave,enter,traffic
0,D01,0,4325,484,NaN
1,D01,5,4707,39868,NaN
2,D01,6,30525,125972,NaN
3,D01,7,51643,317133,NaN
4,D01,8,71401,563664,NaN
...,...,...,...,...,...
515,D27,19,181366,360115,NaN
516,D27,20,138067,288389,NaN
517,D27,21,129992,285953,NaN
518,D27,22,85845,199101,NaN


In [15]:
# calculate traffic

final_dataset['traffic'] = final_dataset['enter'] - final_dataset['leave']

final_dataset

,district,time,leave,enter,traffic
0,D01,0,4325,484,-3841
1,D01,5,4707,39868,35161
2,D01,6,30525,125972,95447
3,D01,7,51643,317133,265490
4,D01,8,71401,563664,492263
...,...,...,...,...,...
515,D27,19,181366,360115,178749
516,D27,20,138067,288389,150322
517,D27,21,129992,285953,155961
518,D27,22,85845,199101,113256


In [16]:
final_dataset.to_csv(
    final_file_path,
    index=False
)